# EDA: Predicting Delivery Time on the Olist E-Commerce Dataset 
## About Dataset:
Brazilian E-Commerce Public Dataset by Olist
Welcome! This is a Brazilian ecommerce public dataset of orders made at Olist Store. The dataset has information of 100k orders from 2016 to 2018 made at multiple marketplaces in Brazil. Its features allows viewing an order from multiple dimensions: from order status, price, payment and freight performance to customer location, product attributes and finally reviews written by customers. We also released a geolocation dataset that relates Brazilian zip codes to lat/lng coordinates.

This is real commercial data, it has been anonymised, and references to the companies and partners in the review text have been replaced with the names of Game of Thrones great houses.


**Project goal:** predict how many days an order takes to reach the customer.
**Data:** [Olist Brazilian E-Commerce (Kaggle)](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce/data), about 100k orders from 2016 to 2018.
**This notebook uses:** `full_data.parquet`, built by `sql/staging.sql` .

## Questions this notebook answers:
1. What does the target (`delivery_time`) look like? Is it skewed? Are there outliers?
2. How many rows have missing coordinates, and what should I do about them?
3. Is delivery time related to distance, state, or time of year?
4. How good is Olist's own estimate compared with the actual delivery time?

In [ ]:
# Setup: make the project root importable so paths come from src/config.py
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "config.py").exists():  
    if ROOT.parent==ROOT:
        raise FileNotFoundError(f"Could not find src/config.py in {Path.cwd()} or any parent folder. "
            "Run this notebook from inside the project.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import FULL_DATA_PARQUET

## Investigating tha dataset

In [3]:
import pandas as pd

df = pd.read_parquet(FULL_DATA_PARQUET)
df.head()

,order_id,purchase_timestamp,estimated_delivery_timestamp,n_items,total_price,total_freight,n_sellers,seller_city,seller_state,seller_latitude,seller_longitude,customer_city,customer_state,customer_latitude,customer_longitude,delivery_time
0,00137e170939bba5a3134e2386413108,2017-11-24 16:50:38,2017-12-18,1,397.00,24.65,1,curitiba,PR,-25.437250,-49.212436,rio de janeiro,RJ,-23.012987,-43.465520,17.098808
1,0017afd5076e074a48f1f1a4c7bac9c5,2017-04-06 22:16:10,2017-05-19,1,809.10,44.29,1,itauna,MG,-20.077863,-44.576134,campo alegre,AL,-9.784323,-36.349941,46.427743
2,0036757472ece3dde52fd4bfd929c90e,2018-07-19 14:58:59,2018-08-20,1,136.99,66.04,1,itaquaquecetuba,SP,-23.486111,-46.366721,bauru,SP,-22.309555,-49.089015,20.982419
3,008eaf9004164883520d0bd32b107261,2017-07-10 20:42:56,2017-08-08,1,135.90,21.44,1,araguari,MG,-18.645099,-48.204993,rio de janeiro,RJ,-22.832605,-43.402809,10.994271
4,00946f674d880be1f188abc10ad7cf46,2017-12-09 19:11:22,2018-01-05,2,199.80,32.92,1,patos de minas,MG,-18.595141,-46.518679,osasco,SP,-23.533560,-46.797805,7.630405


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 96470 entries, 0 to 96469
Data columns (total 16 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   order_id                      96470 non-null  str           
 1   purchase_timestamp            96470 non-null  datetime64[us]
 2   estimated_delivery_timestamp  96470 non-null  datetime64[us]
 3   n_items                       96470 non-null  int64         
 4   total_price                   96470 non-null  float64       
 5   total_freight                 96470 non-null  float64       
 6   n_sellers                     96470 non-null  int64         
 7   seller_city                   96470 non-null  str           
 8   seller_state                  96470 non-null  str           
 9   seller_latitude               96255 non-null  float64       
 10  seller_longitude              96255 non-null  float64       
 11  customer_city                 96470 non

In [11]:
df.isnull().sum()

order_id                          0
purchase_timestamp                0
estimated_delivery_timestamp      0
n_items                           0
total_price                       0
total_freight                     0
n_sellers                         0
seller_city                       0
seller_state                      0
seller_latitude                 215
seller_longitude                215
customer_city                     0
customer_state                    0
customer_latitude               264
customer_longitude              264
delivery_time                     0
dtype: int64

In [5]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
purchase_timestamp,96470,2018-01-01 23:17:43.624411,2016-09-15 12:16:38,2017-09-14 08:56:46.500000,2018-01-20 19:34:43.500000,2018-05-05 18:29:50.250000,2018-08-29 15:00:37,NaN
estimated_delivery_timestamp,96470,2018-01-25 16:58:03.644656,2016-10-04 00:00:00,2017-10-05 00:00:00,2018-02-16 00:00:00,2018-05-28 00:00:00,2018-10-25 00:00:00,NaN
n_items,96470.0,1.14221,1.0,1.0,1.0,1.0,21.0,0.538824
total_price,96470.0,137.040001,0.85,45.9,86.5,149.9,13440.0,209.052608
total_freight,96470.0,22.785798,0.0,13.85,17.17,24.02,1794.96,21.559959
n_sellers,96470.0,1.013901,1.0,1.0,1.0,1.0,5.0,0.123542
seller_latitude,96255.0,-22.794547,-32.079231,-23.612734,-23.425556,-21.757321,-2.501242,2.757061
seller_longitude,96255.0,-47.224221,-63.893565,-48.807104,-46.748071,-46.518679,-34.855616,2.350363
customer_latitude,96206.0,-21.20178,-33.689948,-23.589853,-22.92497,-20.140216,42.184003,5.600154
customer_longitude,96206.0,-46.188752,-72.668881,-48.118347,-46.631968,-43.625086,-8.723762,4.054228


## Key Observations:
- There a missing values in:
1. `seller_latitude` ,`seller_longitude` (215)
2. `customer_latitude`,and `customer_longitude` (264)

    Which will be dropped .

- `total_price`,`delivery_time` seems to be right-skewed as mean is much grater thant the median 
- The maximum and minimum `delivery_time` values are suspicious , worth investigating 